In [42]:
!pip install -q google-genai
!pip install -U google-generativeai
!pip install -q transformers torch sentencepiece

In [ ]:
!pip install -q pandas numpy scikit-learn spacy
!python -m spacy download en_core_web_sm

In [44]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)

C:\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
from google import genai
from getpass import getpass

os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API key: ")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

Enter your Gemini API key:  ········


In [4]:
response = client.models.generate_content(
    model="gemini-3.8-flash",
    contents="Say hello in one sentence."
)
print(response.text)

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Hello, I hope you are having a wonderful day!


In [6]:
import pandas as pd
import numpy as np
import spacy

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

print("Libraries imported successfully!")

Libraries imported successfully!


In [7]:
nlp = spacy.load("en_core_web_sm")
print("spaCy model loaded successfully!")

spaCy model loaded successfully!


In [45]:
print("Loading FLAN-T5...")

MODEL_NAME = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

llm = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME
)

print("✅ FLAN-T5 Loaded")

Loading FLAN-T5...


C:\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\DELL\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Python313\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by defaul

✅ FLAN-T5 Loaded


In [ ]:
train_df = pd.read_csv("banking77_dataset/train.csv")
test_df = pd.read_csv("banking77_dataset/test.csv")

print("Training data:")
display(train_df.head())

print("\nTesting data:")
display(test_df.head())

Training data:


,text,category
0,I am still waiting on my card?,card_arrival
1,What can I do if my card still hasn't arrived ...,card_arrival
2,I have been waiting over a week. Is the card s...,card_arrival
3,Can I track my card while it is in the process...,card_arrival
4,"How do I know if I will get my card, or if it ...",card_arrival



Testing data:


,text,category
0,How do I locate my card?,card_arrival
1,"I still have not received my new card, I order...",card_arrival
2,I ordered a card but it has not arrived. Help ...,card_arrival
3,Is there a way to know when my card will arrive?,card_arrival
4,My card has not arrived yet.,card_arrival


In [9]:
print("Training Shape:", train_df.shape)
print("Testing Shape:", test_df.shape)

print("\nTraining Columns:")
print(train_df.columns)

print("\nMissing Values:")
print(train_df.isnull().sum())

Training Shape: (10003, 2)
Testing Shape: (3080, 2)

Training Columns:
Index(['text', 'category'], dtype='str')

Missing Values:
text        0
category    0
dtype: int64


In [10]:
for i in range(10):
    print("Question:", train_df.iloc[i]["text"])
    print("Intent:", train_df.iloc[i]["category"])
    print("-" * 60)

Question: I am still waiting on my card?
Intent: card_arrival
------------------------------------------------------------
Question: What can I do if my card still hasn't arrived after 2 weeks?
Intent: card_arrival
------------------------------------------------------------
Question: I have been waiting over a week. Is the card still coming?
Intent: card_arrival
------------------------------------------------------------
Question: Can I track my card while it is in the process of delivery?
Intent: card_arrival
------------------------------------------------------------
Question: How do I know if I will get my card, or if it is lost?
Intent: card_arrival
------------------------------------------------------------
Question: When did you send me my new card?
Intent: card_arrival
------------------------------------------------------------
Question: Do you have info about the card on delivery?
Intent: card_arrival
------------------------------------------------------------
Question: W

In [11]:

print("Number of unique intents:", train_df["category"].nunique())
print("\nIntent names:")
print(train_df["category"].unique())

Number of unique intents: 77

Intent names:
<ArrowStringArray>
[                                    'card_arrival',
                                     'card_linking',
                                    'exchange_rate',
                 'card_payment_wrong_exchange_rate',
                        'extra_charge_on_statement',
                          'pending_cash_withdrawal',
                            'fiat_currency_support',
                           'card_delivery_estimate',
                                 'automatic_top_up',
                                 'card_not_working',
                                 'exchange_via_app',
                              'lost_or_stolen_card',
                                        'age_limit',
                                      'pin_blocked',
                          'contactless_not_working',
                   'top_up_by_bank_transfer_charge',
                                   'pending_top_up',
                                  'c

In [12]:
intent_counts = train_df["category"].value_counts()
display(intent_counts)

category
card_payment_fee_charged                            187
direct_debit_payment_not_recognised                 182
balance_not_updated_after_cheque_or_cash_deposit    181
wrong_amount_of_cash_received                       180
cash_withdrawal_charge                              177
                                                   ... 
lost_or_stolen_card                                  82
card_swallowed                                       61
card_acceptance                                      59
virtual_card_not_working                             41
contactless_not_working                              35
Name: count, Length: 77, dtype: int64

In [13]:

def preprocess_text(text):
    doc = nlp(text.lower())
    words = []

    for token in doc:
        if token.is_punct:
            continue
        if token.is_stop:
            continue
        if token.is_space:
            continue

        words.append(token.lemma_)

    return " ".join(words)

In [14]:
text = "I have forgotten my password and I cannot login"
clean_text = preprocess_text(text)

print("Original:")
print(text)

print("\nAfter preprocessing:")
print(clean_text)

Original:
I have forgotten my password and I cannot login

After preprocessing:
forget password login


In [15]:
train_df["clean_text"] = train_df["text"].apply(preprocess_text)
train_df[["text", "clean_text", "category"]].head(10)


,text,clean_text,category
0,I am still waiting on my card?,wait card,card_arrival
1,What can I do if my card still hasn't arrived ...,card arrive 2 week,card_arrival
2,I have been waiting over a week. Is the card s...,wait week card come,card_arrival
3,Can I track my card while it is in the process...,track card process delivery,card_arrival
4,"How do I know if I will get my card, or if it ...",know card lose,card_arrival
5,When did you send me my new card?,send new card,card_arrival
6,Do you have info about the card on delivery?,info card delivery,card_arrival
7,What do I do if I still have not received my n...,receive new card,card_arrival
8,Does the package with my card have tracking?,package card tracking,card_arrival
9,I ordered my card but it still isn't here,order card,card_arrival


In [16]:
test_df["clean_text"] = test_df["text"].apply(preprocess_text)
test_df[["text", "clean_text", "category"]].head()

,text,clean_text,category
0,How do I locate my card?,locate card,card_arrival
1,"I still have not received my new card, I order...",receive new card order week ago,card_arrival
2,I ordered a card but it has not arrived. Help ...,order card arrive help,card_arrival
3,Is there a way to know when my card will arrive?,way know card arrive,card_arrival
4,My card has not arrived yet.,card arrive,card_arrival


In [17]:

X_train = train_df["clean_text"]
y_train = train_df["category"]

X_test = test_df["clean_text"]
y_test = test_df["category"]

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (10003,)
y_train: (10003,)
X_test: (3080,)
y_test: (3080,)


In [18]:
tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=10000
)

X_train_tfidf = tfidf.fit_transform(X_train) #scaling
X_test_tfidf = tfidf.transform(X_test)

print("Training shape:", X_train_tfidf.shape)
print("Testing shape:", X_test_tfidf.shape)

Training shape: (10003, 10000)
Testing shape: (3080, 10000)


In [19]:
model = LogisticRegression(max_iter=2000)
model.fit(X_train_tfidf, y_train)

print("Model training completed!")

Model training completed!


In [20]:
y_pred = model.predict(X_test_tfidf)
print(y_pred[:20])

['country_support' 'card_arrival' 'card_arrival' 'card_arrival'
 'card_arrival' 'getting_spare_card' 'card_arrival'
 'card_delivery_estimate' 'card_arrival' 'card_arrival' 'card_arrival'
 'card_delivery_estimate' 'card_arrival' 'card_arrival' 'card_arrival'
 'card_delivery_estimate' 'card_delivery_estimate' 'order_physical_card'
 'card_arrival' 'card_arrival']


In [21]:
accuracy = accuracy_score(y_test, y_pred)
print("Model Accuracy:", round(accuracy * 100, 2), "%")

Model Accuracy: 81.82 %


In [22]:
print(classification_report(y_test, y_pred))

                                                  precision    recall  f1-score   support

                           Refund_not_showing_up       0.90      0.93      0.91        40
                                activate_my_card       1.00      0.93      0.96        40
                                       age_limit       0.97      0.95      0.96        40
                         apple_pay_or_google_pay       0.97      0.97      0.97        40
                                     atm_support       0.89      0.85      0.87        40
                                automatic_top_up       1.00      0.93      0.96        40
         balance_not_updated_after_bank_transfer       0.64      0.70      0.67        40
balance_not_updated_after_cheque_or_cash_deposit       0.84      0.93      0.88        40
                         beneficiary_not_allowed       0.87      0.85      0.86        40
                                 cancel_transfer       0.89      0.97      0.93        40
         

In [23]:
def predict_intent(message):
    clean_message = preprocess_text(message)
    message_vector = tfidf.transform([clean_message])
    prediction = model.predict(message_vector)[0]
    probabilities = model.predict_proba(message_vector)
    confidence = np.max(probabilities)

    return prediction, confidence

In [24]:
message = "My card has not arrived yet"

intent, confidence = predict_intent(message)

print("Customer:", message)
print("Predicted Intent:", intent)
print("Confidence:", round(confidence * 100, 2), "%")

Customer: My card has not arrived yet
Predicted Intent: card_arrival
Confidence: 35.02 %


In [40]:
def generate_response(
    message,
    intent,
    confidence
):
    prompt = f"""
You are SmartBank AI Assistant.

Customer Message:
{message}

Detected Intent:
{intent}

Confidence Score:
{round(confidence*100,2)}%

Generate a short professional banking support response.

Rules:
- Maximum 100 words
- Be helpful
- Do not ask for OTP
- Do not ask for PIN
- Do not ask for passwords
"""
    response = client.models.generate_content(model="gemini-3.8-flash",contents=prompt)
    return response.text
    

In [48]:
def generate_response_huggingface(
    message,
    intent,
    confidence
):

    prompt = f"""
You are a professional banking assistant.

Customer Message:
{message}

Detected Intent:
{intent}

Confidence:
{round(confidence*100,2)}%

Generate a short professional banking response.

Rules:
- Maximum 2 sentences
- Friendly
- Professional
- Never ask for OTP
- Never ask for PIN
- Never ask for Password

Response:
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    )

    outputs = llm.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

    reply = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    return reply



In [26]:
questions = [
    "My card has not arrived",
    "I forgot my PIN",
    "Why was I charged for withdrawing cash?",
    "I don't recognize this transaction",
    "My transfer is still pending",
    "I want to change my PIN"
]

for question in questions:
    intent, confidence = predict_intent(question)

    print("Customer:", question)
    print("Intent:", intent)
    print("Confidence:", round(confidence * 100, 2), "%")
    print("-" * 60)

Customer: My card has not arrived
Intent: card_arrival
Confidence: 35.02 %
------------------------------------------------------------
Customer: I forgot my PIN
Intent: pin_blocked
Confidence: 48.24 %
------------------------------------------------------------
Customer: Why was I charged for withdrawing cash?
Intent: cash_withdrawal_charge
Confidence: 73.68 %
------------------------------------------------------------
Customer: I don't recognize this transaction
Intent: compromised_card
Confidence: 10.92 %
------------------------------------------------------------
Customer: My transfer is still pending
Intent: pending_transfer
Confidence: 95.09 %
------------------------------------------------------------
Customer: I want to change my PIN
Intent: change_pin
Confidence: 95.18 %
------------------------------------------------------------


In [27]:
responses = {
    "card_arrival": "Your card delivery is being checked. Please check the expected delivery date in your app.",
    "card_delivery_estimate": "Standard physical cards usually take 3 to 5 business days to arrive locally, and up to 2 weeks internationally.",
    "card_not_received": "If your card has not arrived after the estimated window, please report it as lost in the app so we can ship a replacement.",
    "pin_blocked": "If your PIN is blocked due to 3 incorrect attempts, you can easily unblock it in the app under Card Settings -> Security -> Unblock PIN.",
    "change_pin": "You can change your PIN at any supported ATM by selecting 'PIN Services' after inserting your physical card.",
    "cash_withdrawal_not_recognised": "If you don't recognize a cash withdrawal, freeze your card instantly in the app and contact our fraud team immediately.",
    "cash_withdrawal_charge": "We do not charge for the first 5 ATM withdrawals per month. After that, a standard 2% fee may apply depending on your plan.",
    "pending_transfer": "Transfers can remain pending while banking systems clear them. Most local transfers clear instantly, while international ones take 1-3 business days.",
    "card_payment_not_recognised": "This card payment is not recognized. Freeze your card in the app and submit a transaction dispute form if you suspect fraud.",
    "lost_or_stolen_card": "If your card is lost or stolen, please freeze it instantly in the SmartBank app and request a replacement card.",
    "declined_cash_withdrawal": "Cash withdrawals can be declined due to insufficient funds, an incorrect PIN, reaching daily ATM limits, or your card being frozen.",
    "top_up_failed": "If your top-up failed, please check that your source card has sufficient funds, is active for online transactions, or try using another card/payment method.",
    "country_support": "SmartBank is currently supported across the US, UK, EEA countries, and Switzerland. We are constantly expanding!"
}

In [28]:
def banking_chatbot(message):
    # Basic cleaning for greetings
    greetings = ["hi", "hello", "hey", "greetings", "good morning", "good afternoon"]
    if message.strip().lower() in greetings:
        print("🤖 SmartBank: Hello! I am your SmartBank digital assistant. How can I help you today?")
        return

    intent, confidence = predict_intent(message)

    # Smart thresholds without extra debug logging
    if confidence >= 0.45:
        response = responses.get(
            intent,
            f"I've identified your request regarding '{intent.replace('_', ' ')}'. How can I assist you further?"
        )
    elif confidence >= 0.20:
        friendly_intent = intent.replace('_', ' ')
        response = f"It sounds like you might be asking about '{friendly_intent}'. " + responses.get(intent, "Please contact our support desk.")
    else:
        response = "I'm sorry, I couldn't quite understand your request. Could you please rephrase it with more details?"

    print(f"🤖 SmartBank: {response}")

In [29]:
banking_chatbot("I don't know where my card is")

🤖 SmartBank: I'm sorry, I couldn't quite understand your request. Could you please rephrase it with more details?


In [30]:
banking_chatbot("I forgot my PIN")

🤖 SmartBank: If your PIN is blocked due to 3 incorrect attempts, you can easily unblock it in the app under Card Settings -> Security -> Unblock PIN.


In [31]:
banking_chatbot("Someone withdrew money from my account")

🤖 SmartBank: It sounds like you might be asking about 'declined cash withdrawal'. Cash withdrawals can be declined due to insufficient funds, an incorrect PIN, reaching daily ATM limits, or your card being frozen.


In [32]:
# Final Simplified Interactive Chatbot Loop using local model
print("Starting local SmartBank Chatbot. Type 'exit' to quit.")
while True:
    user_input = input("\nYou: ")
    if user_input.lower() == "exit":
        print("SmartBank: Thank you! Goodbye.")
        break
    banking_chatbot(user_input)

Starting local SmartBank Chatbot. Type 'exit' to quit.



You:  HI


🤖 SmartBank: Hello! I am your SmartBank digital assistant. How can I help you today?



You:  I want to change my pin


🤖 SmartBank: You can change your PIN at any supported ATM by selecting 'PIN Services' after inserting your physical card.


...

🤖 SmartBank: If your PIN is blocked due to 3 incorrect attempts, you can easily unblock it in the app under Card Settings -> Security -> Unblock PIN.



You:  i cannot withdraw money from my account


🤖 SmartBank: It sounds like you might be asking about 'declined cash withdrawal'. Cash withdrawals can be declined due to insufficient funds, an incorrect PIN, reaching daily ATM limits, or your card being frozen.



You:  exit


SmartBank: Thank you! Goodbye.


In [51]:
def chatbot(message):
    # Check for simple greeting matches
    greetings = ["hi", "hello", "hey", "greetings"]
    if message.strip().lower() in greetings:
        return """
----------------------------------
Response:
Hello! Welcome to SmartBank. How can I assist you with your banking services today?
----------------------------------
"""

    intent, confidence = predict_intent(
        message
    )

    if confidence < 0.20:
        return (
            "Sorry, I could not understand "
            "your request clearly."
        )

    response = generate_response_huggingface(
        message,
        intent,
        confidence
    )

    return f"""
----------------------------------
Intent     : {intent}
Confidence : {round(confidence*100,2)}%

Response:
{response}
----------------------------------
"""

In [52]:
print("\n===================================")
print(" SMARTBANK NLP + GENAI/Huggingface CHATBOT ")
print(" Type 'exit' to quit")
print("===================================\n")

while True:
    try:
        user_input = input("You: ")
    except (KeyboardInterrupt, EOFError):
        break

    if user_input.lower() == "exit":
        print("\nSmartBank: Goodbye!")
        break

    result = chatbot(user_input)
    print(result)


 SMARTBANK NLP + GENAI/Huggingface CHATBOT 
 Type 'exit' to quit



You:  Hi



----------------------------------
Response:
Hello! Welcome to SmartBank. How can I assist you with your banking services today?
----------------------------------



You:  I lost my card



----------------------------------
Intent     : lost_or_stolen_card
Confidence : 68.68%

Response:
No, I don't want to ask for PIN
----------------------------------



You:  I forgot my pin



----------------------------------
Intent     : pin_blocked
Confidence : 48.24%

Response:
Never ask for PIN
----------------------------------



You:  exit



SmartBank: Goodbye!
